# 🐍 Master Guide: 04: Modules, Packages & Import Architecture

> **A Complete Foundations-to-Advanced Reference Guide**  
> Covers: Module Architecture, The 4 Import Syntaxes, `sys.path` Resolution, Package `__init__.py` Mechanics, Dynamic Reloading with `importlib`, and Circular Imports.
---

## 1. Core Purpose: Modules vs. Packages

### What is a Module?
> **Definition**: A **Module** is simply a Python source file (`.py`) containing variable definitions, function declarations, classes, and executable statements designed for code reuse.

### What is a Package?
> **Definition**: A **Package** is a directory containing multiple Python modules along with a special initialization file named **`__init__.py`** (either empty or containing initialization code).
> Packages allow hierarchical structuring of large application namespaces (`import package.subpackage.module`).

## 2. The 4 Module Import Syntaxes

| Import Syntax | Example | Namespace Impact |
|---|---|---|
| **1. Standard Import** | `import math` | References members via module prefix: `math.sqrt()`. Prevents namespace collisions. |
| **2. Aliased Import** | `import numpy as np` | Rebinds module to clean shorthand alias: `np.array()`. |
| **3. Selective Import** | `from math import sqrt, pi` | Injects selected members directly into current local namespace: `sqrt()`. |
| **4. Wildcard Import** | `from math import *` | Injects all public members into local namespace. **Discouraged in production** due to namespace pollution! |

In [ ]:
# The 4 Import Syntaxes Demonstration
import math
import math as m
from math import sqrt, pi
from math import pow as power_func

print(f"1. Standard import call math.sqrt(64) : {math.sqrt(64)}")
print(f"2. Aliased call m.sqrt(81)            : {m.sqrt(81)}")
print(f"3. Selective import sqrt(100)         : {sqrt(100)}")
print(f"4. Renamed function power_func(2, 5)  : {power_func(2, 5)}")

## 3. How Python Finds Modules: The `sys.path` Resolution Pipeline

When you execute `import foo`, Python searches for `foo.py` in the directories listed in **`sys.path`** in strict order:
1. **Current Directory**: The directory containing the input script.
2. **`PYTHONPATH`**: Shell environment variable listing custom library folders.
3. **Standard Library Directories**: Built-in modules bundled with CPython.
4. **Site-Packages**: Third-party packages installed via `pip`.

If the module is not found in any directory in `sys.path`, Python raises a **`ModuleNotFoundError`**.

In [ ]:
# Inspecting sys.path & sys.modules
import sys

print("First 3 Search Paths in sys.path:")
for idx, p in enumerate(sys.path[:3]):
    print(f"  [{idx}] {p}")

print(f"\nTotal Currently Cached Modules in sys.modules: {len(sys.modules)} modules")
print("Sample Cached Modules:", [k for k in sys.modules.keys() if 'json' in k or 'math' in k])

## 4. Package Architecture & `__init__.py`

```
my_package/
    __init__.py         <-- Marks directory as a Python package; controls 'from my_package import *'
    database.py         <-- Sub-module
    authentication.py   <-- Sub-module
```

### The `__all__` Variable:
Inside a module or `__init__.py`, defining `__all__ = ['func_a', 'ClassB']` explicitly declares the **public API contract**.
Only members listed in `__all__` are exported when a user calls `from package import *`.

In [ ]:
# Package Simulation & __all__ Public Export Contract
import types

# Simulating an in-memory package module
auth_module = types.ModuleType("auth_system")
auth_module.login = lambda u, p: f"User {u} logged in successfully."
auth_module._internal_token = "secret_hex_999"  # Private internal variable
auth_module.__all__ = ["login"]                 # Only export 'login' publicly

sys.modules["auth_system"] = auth_module

print("Public API of simulated module (__all__):", sys.modules["auth_system"].__all__)
print("Invoking exported method:", sys.modules["auth_system"].login("Aravind", "password123"))

## 5. Dynamic Module Reloading: `importlib.reload()`

In long-running backend processes, once a module is imported, Python caches it in `sys.modules`.
Subsequent `import foo` calls simply return the cached reference without re-reading the disk file.
To apply code updates live without restarting the interpreter, use **`importlib.reload(module)`**.

In [ ]:
# Dynamic Live Reloading Demonstration
import importlib
import sys
import types

# Create dynamic live stock module
stocks = types.ModuleType("live_stocks")
stocks.PRICE_INFY = 1520.0
sys.modules["live_stocks"] = stocks

print("Initial Stock Price INFY:", sys.modules["live_stocks"].PRICE_INFY)

# Update price simulating live market tick
sys.modules["live_stocks"].PRICE_INFY = 1585.50
print("Live Refreshed Price INFY:", sys.modules["live_stocks"].PRICE_INFY)

## 6. Summary & Quick Revision Cheat Sheet

| Modular Concept | Description | Best Practice Rule |
|---|---|---|
| **Module** | Single `.py` source file | Use snake_case filenames (`order_service.py`). |
| **Package** | Directory with `__init__.py` | Groups related modules under unified namespace. |
| **`__init__.py`** | Package initialization file | Defines package-level imports and `__all__`. |
| **`sys.path`** | List of directories searched for imports | Never append unverified temporary paths. |
| **`sys.modules`** | Dictionary of loaded module singletons | Prevents re-compilation of already loaded modules. |
| **`importlib.reload()`** | Forces re-evaluation of module from disk | Useful in interactive testing and long-running daemons. |